# 04 — Outcome Prediction (Hired vs Not Hired)

Implements the benchmark **preprocessing** methodology from:

> Weytjens, H., & De Weerdt, J. (2021). *Creating Unbiased Public Benchmark Datasets
> with Data Leakage Prevention for Predictive Process Monitoring.*
> AI4BPM Workshop @ BPM 2021. arXiv:2107.01905.

**Key methodology steps applied (paper references):**

| Step | §ref | Description |
|------|------|-------------|
| Remove chronological outliers / long cases | §5.2, §5.7 | Keep cases within 95th-percentile duration |
| Strict temporal split | §5.5 | Train = cases **completed** before `sep_time` only |
| Test = last 20% | §5.4 | `sep_time` = first-event timestamp of 80th-percentile case |
| Debias test start | §5.6 | Straddler cases (started before / ended after `sep_time`) added to test |
| Validation set | §4.3 | 20% latest-starting train cases; used for early stopping |
| Metric | §5.1 | AUC-ROC (robust to class imbalance) |

**Models:**
- **LightGBM** and **XGBoost** — Boolean prefix encoding, prefix sweep k=[1,3,5,10,20,all]
- **CNN** — paper's exact architecture (§4.3): Embedding → 2×(Conv1D(40,3)+MaxPool) → Dense(100)→Dense(10)→Dense(1)
- **LSTM** — same sequence input, LSTM(100) encoder

## 1. Imports

In [47]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os, random
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

import pm4py as pm
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(RANDOM_STATE)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

Device : cuda
GPU    : NVIDIA GeForce RTX 3060 Laptop GPU


## 2. Configuration

In [48]:
# ── Paper hyperparameters (Weytjens & De Weerdt, 2021) ────────────────────────
TEST_FRACTION    = 0.20    # Last 20% of cases by case_start → test set (§5.4)
MAX_DUR_PCT      = 0.95    # Remove top 5% longest cases to preserve training set (§5.7)
BATCH_SIZE       = 2048    # §4.3
PATIENCE         = 30      # Early stopping patience (§4.3)
CNN_FILTERS      = 40      # Conv1D kernels per layer (§4.3)
CNN_KERNEL       = 3       # Kernel size (§4.3)
DENSE_1          = 100     # First dense layer size (§4.3)
DENSE_2          = 10      # Second dense layer size (§4.3)

# ── Adaptation parameters ─────────────────────────────────────────────────────
MAX_SEQ_LEN      = 20      # Paper uses 10; adapted to cover recruiting trace lengths
MAX_EPOCHS       = 150
LEAKY_THRESHOLD  = 0.85
PREFIX_LENGTHS   = [1, 3, 5, 10, 20, 'all']

print('Config loaded.')

Config loaded.


## 3. Load & Clean

In [49]:
dtype_dict = {
    'hired': 'Int8', 'Rejected': 'Int8', 'CW': 'Int8', 'Evergreen': 'Int8',
    'Region': 'Int16', 'Country': 'Int16',
    'Job Family': 'Int16', 'Job Family Group': 'Int16',
}
df = pd.read_csv(
    'data/event_log_consolidated.csv',
    low_memory=False, dtype=dtype_dict, parse_dates=['timestamp']
)
df["Recruiting Agency"] = df["Recruiting Agency"].fillna(0)
df = df[~df['Step'].str.match(r'^\d+$', na=False)].dropna(subset=['Step']).copy()
df.sort_values(['Case_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)
df = pm.format_dataframe(df, case_id='Case_id', activity_key='Step', timestamp_key='timestamp')
df.rename(columns = {"Completed By":"org:resource"}, inplace = True)
print(f'Events     : {len(df):,}')
print(f'Cases      : {df["case:concept:name"].nunique():,}')
print(f'Activities : {df["concept:name"].nunique()}')

Events     : 511,072
Cases      : 268,304
Activities : 65


## 2. Paper Preprocessing (Weytjens & De Weerdt, 2021)

### 2a. Remove Long Cases (§5.7)

Extremely long cases would "kill" the training set when debiasing the end of the dataset,
because the entire yellow zone would span the maximum case duration. The paper proposes
removing up to 5% of the longest-lasting cases to find the training-set-maximising cutoff.

In [50]:
case_times = df.groupby('case:concept:name').agg(
    case_start=('time:timestamp', 'min'),
    case_end  =('time:timestamp', 'max'),
).sort_values('case_start')

case_times['duration_days'] = (
    (case_times['case_end'] - case_times['case_start']).dt.total_seconds() / 86400
)

max_dur = case_times['duration_days'].quantile(MAX_DUR_PCT)
n_before = len(case_times)
case_times = case_times[case_times['duration_days'] <= max_dur].copy()
n_after = len(case_times)
df = df[df['case:concept:name'].isin(case_times.index)].copy()

print(f'Duration cutoff : {max_dur:.1f} days (95th percentile)')
print(f'Cases removed   : {n_before - n_after:,} ({(n_before-n_after)/n_before:.1%})')
print(f'Cases remaining : {n_after:,}')

Duration cutoff : 86.9 days (95th percentile)
Cases removed   : 13,415 (5.0%)
Cases remaining : 254,889


### 2b. Remove Leaky activities

Activities with a hire rate ≥ 0.85 on the **fit set** are post-decision steps that
encode the outcome (background checks, offer documents, etc.). Removed before any
feature extraction to prevent label leakage.

In [51]:
# ── Preliminary temporal split to identify fit cases ────────────────────────
# Leaky activities must be computed from the fit set only (Weytjens §5.3).
# We use a preliminary sep_time (same logic as the final split) for this.
_n       = len(case_times)
_sep     = case_times.iloc[int(_n * (1 - TEST_FRACTION))]['case_start']
_fit_ids = case_times[case_times['case_end'] < _sep].index

def get_leaky_activities(df, fit_ids):
    """
    Returns activities whose case-level hire rate >= LEAKY_THRESHOLD on the fit set.
    Hire rate per activity = fraction of fit cases *containing* that activity that are hired.
    Uses drop_duplicates to count each (case, activity) pair once regardless of repetition.
    """
    fit    = df[df['case:concept:name'].isin(fit_ids)].copy()
    fit_ca = fit.drop_duplicates(['case:concept:name', 'concept:name'])
    act_hire_rate = (
        fit_ca.groupby('concept:name')
              .apply(lambda g: g['hired'].astype(float).mean())
    )
    return set(act_hire_rate[act_hire_rate >= LEAKY_THRESHOLD].index)

remove_leaky = False
if remove_leaky:
    leaky_acts = get_leaky_activities(df, _fit_ids)
    # Remove ROWS with leaky activities — keeps all cases but strips post-decision events
    df = df[~df['concept:name'].isin(leaky_acts)].copy()

    # Recompute case_times: leaky removal can shift a case's effective last-event timestamp,
    # which affects the strict-end-time boundary used in split_train_test.
    case_times = df.groupby('case:concept:name').agg(
        case_start=('time:timestamp', 'min'),
        case_end  =('time:timestamp', 'max'),
    ).sort_values('case_start')
    case_times['duration_days'] = (
        (case_times['case_end'] - case_times['case_start']).dt.total_seconds() / 86400
    )

    print(f'Leaky activities removed : {len(leaky_acts)}')
    print(sorted(leaky_acts))
    print(f'Activities remaining     : {df["concept:name"].nunique()}')
    print(f'Cases remaining          : {df["case:concept:name"].nunique():,}')


### 3 Aggregate encoding. 

In [56]:
import pandas as pd

def aggregation_encode(df, case_col, prefix_col, cat_event_cols, num_event_cols, case_attr_cols, target_col=None):
    """
    Aggregation encoding for prefixed event logs (slide logic):
      - cat_event_cols : categorical event attrs → cumulative count of each value in prefix
      - num_event_cols : numerical event attrs   → cumulative sum over prefix
      - case_attr_cols : static per case         → passed through as-is
    """
    df = df.sort_values([case_col, prefix_col]).copy()

    # --- Categorical event attrs: one-hot → cumsum within case ---
    cat_dummies = pd.get_dummies(df[cat_event_cols], dtype=int)
    cat_encoded = cat_dummies.groupby(df[case_col].values).cumsum()

    # --- Numerical event attrs: cumsum within case ---
    num_encoded = (
        df[num_event_cols]
        .groupby(df[case_col].values)
        .cumsum()
        .rename(columns={c: f"sum_{c}" for c in num_event_cols})
    )

    # --- Assemble ---
    base_cols = [case_col, prefix_col] + case_attr_cols + (target_col if target_col else [])
    result = pd.concat(
        [df[base_cols].reset_index(drop=True),
         cat_encoded.reset_index(drop=True),
         num_encoded.reset_index(drop=True)],
        axis=1
    )
    return result

grouping_cols = ['Recruiting Agency', 'Region',
       'Country', 'Job Family', 'Job Family Group','CW',
       'Evergreen',]
target_col = ["hired"]
case_col = ["case:concept:name"]
time_col = ["time:timestamp"]
agg_cols =  ['org:resource', 'concept:name', 'prefix', '@@case_index']

df["prefix"] = df.groupby(case_col).cumcount() + 1
df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True)
df['time_diff'] = df.groupby("case:concept:name")["time:timestamp"].diff().dt.total_seconds().div(3600).fillna(0)
df.head()
# ── Apply to your df ──────────────────────────────────────────────
encoded_df = aggregation_encode(
    df=df,
    case_col      = "case:concept:name",
    prefix_col    = "prefix",
    cat_event_cols= ["concept:name", "org:resource"],   # → Act_X / Res_X counts
    num_event_cols= ["time_diff"],                       # → sum_time_diff
    case_attr_cols= grouping_cols,
    target_col    = target_col,
)

encoded_df.head()

,case:concept:name,prefix,Recruiting Agency,Region,Country,Job Family,Job Family Group,CW,Evergreen,hired,...,concept:name_Review Decision,concept:name_Review Documents,concept:name_Review Offer Letter,concept:name_Schedule Interview,concept:name_Screen,concept:name_Screen Candidate,concept:name_Set Background Check Status,concept:name_Start,concept:name_Workday Interview,sum_time_diff
0,000086ee80c8 - 10fb9be7d6ea,1,0,4,15,57,16,0,0,0,...,0,0,0,0,0,0,0,1,0,0.000000
1,000086ee80c8 - 10fb9be7d6ea,2,0,4,15,57,16,0,0,0,...,0,0,0,0,0,0,0,1,0,0.012447
2,000086ee80c8 - 10fb9be7d6ea,3,0,4,15,57,16,0,0,0,...,0,0,0,1,0,0,0,1,0,24.364193
3,000086ee80c8 - 10fb9be7d6ea,4,0,4,15,57,16,0,0,0,...,0,0,0,1,0,0,0,1,0,168.385300
4,000086ee80c8 - 10fb9be7d6ea,5,0,4,15,57,16,0,0,0,...,0,0,0,2,0,0,0,1,0,168.411729


In [58]:
encoded_df.drop(["prefix"], axis = 1, inplace = True)

In [67]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# --- Config ---
TARGET = "hired"
ID_COL = "case:concept:name"
CAT_FEATURES = ['Recruiting Agency', 'Region', 'Country',
       'Job Family', 'Job Family Group', 'CW', 'Evergreen', 'org:resource']  # specify your categorical columns here

# --- Prep ---
X = encoded_df.drop(columns=[TARGET, ID_COL])
y = encoded_df[TARGET]

cat_indices = [X.columns.get_loc(c) for c in CAT_FEATURES]

# Compute class weight ratio for imbalance
neg, pos = (y == 0).sum(), (y == 1).sum()
scale_pos_weight = neg / pos
print(f"Class ratio (neg/pos): {scale_pos_weight:.2f}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.01,
    depth=6,
    cat_features=cat_indices,
    eval_metric="AUC",
    early_stopping_rounds=150,
    scale_pos_weight=scale_pos_weight,
    random_seed=42,
    verbose=100,
)

model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
)

# --- Eval ---
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")
# --- Feature importance ---
feat_imp = pd.Series(model.get_feature_importance(), index=X.columns).sort_values(ascending=False)
print(feat_imp.head(15))

Class ratio (neg/pos): 6.11
0:	test: 0.9331354	best: 0.9331354 (0)	total: 226ms	remaining: 5m 39s
100:	test: 0.9833561	best: 0.9833561 (100)	total: 21.3s	remaining: 4m 54s
200:	test: 0.9841708	best: 0.9841708 (200)	total: 43s	remaining: 4m 37s
300:	test: 0.9850859	best: 0.9850859 (300)	total: 1m 3s	remaining: 4m 12s
400:	test: 0.9856643	best: 0.9856643 (400)	total: 1m 27s	remaining: 4m
500:	test: 0.9861823	best: 0.9861823 (500)	total: 1m 52s	remaining: 3m 43s
600:	test: 0.9865891	best: 0.9865891 (600)	total: 2m 18s	remaining: 3m 26s
700:	test: 0.9869238	best: 0.9869238 (700)	total: 2m 44s	remaining: 3m 7s
800:	test: 0.9871631	best: 0.9871631 (800)	total: 3m 6s	remaining: 2m 43s
900:	test: 0.9873888	best: 0.9873888 (900)	total: 3m 29s	remaining: 2m 19s
1000:	test: 0.9876163	best: 0.9876163 (1000)	total: 3m 51s	remaining: 1m 55s
1100:	test: 0.9878353	best: 0.9878353 (1100)	total: 4m 14s	remaining: 1m 32s
1200:	test: 0.9880079	best: 0.9880079 (1200)	total: 4m 36s	remaining: 1m 8s
1300:	te